# CobraBox Use-Case : Directed Connectivity for Localizing Epileptogenicity (NB #1: Data preparation & Connectivity estimation)

Authors: *[COBRA group](https://cobra.cs.cas.cz), Institute of Computer Science, The Czech Academy of Sciences*

<div align="left">
<img src="Images/Logo_CAS_ICS.png" align="left" width="254" alt="logo ICS">
</div>


<br>
<br>

---------------------

This is the first of the two notebooks that together form a complete, reproducible example of using [CobraBox](https://github.com/cobragroup/cobrabox) on an intracranial EEG (iEEG) dataset:

1. ***Data preprocessing and connectivity estimation (this notebook)*** turns the raw recordings of the [interictal iEEG Zurich dataset](https://openneuro.org/datasets/ds003498/versions/1.1.1) into clean, segmented, analysis-ready data, computes directed connectivity using directed transfer function (DTF), and calculates the inward and outward strength of each network node (electrode).
2. ***Analysis and statistics*** runs the analysis and statistics to localize the epileptogenic tissue and predict the surgical outcome for the patients in the cohort.

Both notebooks follow the pipeline of the accompanying study (Stergiadis C, Halliday DM, Kazis D, Klados MA. *High-frequency directed networks can identify epileptogenic tissue and predict surgical outcome in drug-resistant epilepsy*. Epilepsy Research. 2026;226:107838. doi: [10.1016/j.eplepsyres.2026.107838](https://doi.org/10.1016/j.eplepsyres.2026.107838)).

<span style="color:darkred">
IMPORTANT: To keep this notebook readable, a few routine steps that are <b>not</b> part of CobraBox itself — loading the patient metadata, notch filtering, building the bipolar montage, selecting the data segments, and reading/writing intermediate files — are collected in a small local helper module, <code>local_utils.py</code>. Every time such a helper is used it is called explicitly as <code>local_utils.&lt;function&gt;()</code> and the step is explained in the surrounding text, so nothing important is hidden: you are encouraged to open <code>local_utils.py</code> to see exactly what each one does. Note the relative import <code>import local_utils</code> (not <code>from local_utils import *</code>).
</span><br><br><b>Important:</b> This notebook is <b>self-contained</b>: you do not need to have run 01_data_exporation to follow it.

### Contents of this book

1. Loading the dataset
2. Patient metadata
3. The preprocessing steps (notch, bipolar montage, segmentation)
4. Running the pipeline for all subjects
5. Verifying the output
6. Directed connectivity (DTF)
7. Inward and outward strength
8. Where to go next …

### Import dependencies

Running this notebook requires ***python*** (>=3.11) together with ***xarray*** (>2026.2.0) and
***cobrabox*** (>=X.Y). These should have been installed together with ***cobrabox***; please visit
the [CobraBox documentation](#) for installation instructions. The routine preprocessing helpers used
below live in the local module ***local_utils.py*** that ships alongside these notebooks.

In [ ]:
## IMPORT THE PACKAGES NEEDED TO RUN THE NOTEBOOK
# Python standard library imports
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Third-party imports
import cobrabox as cb

# Local libraries and modules
import local_utils

### Configuration for this notebook

We point CobraBox at a local `data/` folder, list the subjects, and fix the two segmentation
parameters used by the study: **20** segments of **3 seconds** each per subject (Stergiadis et al.,
2026, §2.4).

In [ ]:
# Set the local path to store the data
cb.set_dataset_dir(Path(".") / "data", persist=False)

DATASET_NAME = "zurich_ieeg_clean"

# The 20 subjects of the Zurich iEEG dataset
SUBJECTS = [f"sub-{i:02d}" for i in range(1, 21)]

# Where the analysis-ready segments will be written
DATA_DIR = Path("data")
SEGMENTS_DIR = DATA_DIR / "segments"
SEGMENTS_DIR.mkdir(parents=True, exist_ok=True)

# Segmentation parameters (Stergiadis et al., 2026, section 2.4)
SEGMENT_DURATION = 3  # seconds per segment
N_SEGMENTS = 20  # random segments per subject

### What this notebook produces

In short, we first take a public **intracranial EEG (iEEG)** dataset, which is brain activity recorded from electrodes placed directly on top or deep inside the brain of people with drug-resistant epilepsy. We then turn each raw recording into a handful of short, clean, analysis-ready segments saved to disk. These segments are subsequently used to run the directed connectivity analysis followed in (Stergiadis et al., 2026).

01_data_exploration.ipynb explored a single subject. Here we run an analysis **for all the subjects of the iEEG dataset (n=20)** following the approach of (Stergiadis et al., 2026) and
save the results to disk, so that 03_analysis.ipynb can go straight to the analysis and statistics.

For each subject the pipeline steps are:

1. **Fetch** the *clean* version of the recordings using `cb.load_dataset(..., clean=True)`.
2. **Notch-filter** each run at 50 Hz to remove power-line noise (§2.3).
3. **Re-reference** to a **single-spacing bipolar montage** over the valid channel pairs (§2.3).
4. **Extract 20 random 3-second segments** (§2.4).
5. **Save** the segments as an `(segment × space × time)` array in `data/segments/`.
6. **Compute DTF** in each 3-second segment and aversage across the segments per frequency band
7. **Calculate inward and outward strength** using the averaged matrix

In everyday terms: step 1 downloads the data; step 2 removes the 50 Hz hum that mains electricity leaks into the recording; step 3 re-expresses each channel relative to its neighbour so we capture *local* brain activity rather than a shared background; step 4 cuts the recording into twenty short 3-second stretches; step 5 writes the result out to disk; step 6 load the segments and quantifies the directionality and strength of the connections between the implanted areas; step7 calculates and saves the strength of incoming and outgoing connections in each electrode

We load the dataset in **`clean` mode** (`clean=True`): a curated version of `zurich_ieeg` whose format and metadata are identical to the full dataset, but whose channels are trimmed to the clean set (any discrepancies between raw data and metadata have been fixed and only electrodes with known resection status are included). Because of this, the notebook needs no separate channel-exclusion step.


## 1. Loading the dataset

Before anything else, let us actually load the data and look at it, so it is clear what we are working with. 

CobraBox fetches the dataset for us with `cb.load_dataset()`. We pass three things:

- `"zurich_ieeg_clean"` — the dataset's name in CobraBox (the **clean** version described above curated channels, metadata reconciled).
- `subset=["sub-01"]` — which subject(s) to download. The full dataset is large (~60 GB), so here we fetch a single subject just to inspect it; the batch loop later goes over all 20 patients (see 4. Running the pipeline for all subjects).

What comes back is a small container of one or more **runs**. Each run is one continuous recording, stored as an array with a `space` axis (the electrode contacts, i.e. the channels) and a `time` axis (the samples), together with its **sampling rate** (the number of samples recorded per second). The cell below loads `sub-01` and prints these basic properties.

In [ ]:
# Fetch and load one subject (clean version) just to inspect it
ds = cb.load_dataset(DATASET_NAME, subset=["sub-01"])

print(f"Loaded {len(ds)} run(s) for sub-01\n")
for run_idx, item in enumerate(ds):
    fs = item.sampling_rate
    n_samples = item.data.sizes["time"]
    n_channels = item.data.sizes["space"]
    duration = n_samples / fs
    print(f"  Run {run_idx}: {fs:.0f} Hz, {duration:.1f} s, {n_channels} channels")

## 2. Patient metadata

Each subject comes with a small record: the surgical **outcome** (ILAE grade) and the list of
**resected** bipolar pairs. We load it with the helper
`local_utils.load_patient_info()` (which reads the companion CSV — see `local_utils.py`) and prints a
short summary of the cohort.

In [ ]:
patient_info = local_utils.load_patient_info()

header = f"{'subject':<10} {'outcome':<10} {'n_resected':>12} {'n_excluded':>12}"
print(header)
print("-" * len(header))
for sub_id, info in patient_info.items():
    print(
        f"{sub_id:<10} {info['outcome']:<10} "
        f"{len(info['resected']):>12} {len(info['pipeline_exclusions']):>12}"
    )

## 3. The preprocessing steps

The three signal-processing steps below are standard iEEG preprocessing rather than CobraBox features,
so their implementations live in `local_utils.py`. We explain what each does here and call them
explicitly; open `local_utils.py` to see the exact filter design and montage construction.

**4.1 Notch filter (50 Hz).** European mains introduce a 50 Hz artefact. We remove it with a narrow
zero-phase IIR notch on the unipolar signals, before montaging. A *notch* filter suppresses one narrow frequency — here the 50 Hz power-line hum — while leaving neighbouring frequencies essentially untouched; *zero-phase* means it introduces no time shift into the signal.

**4.2 Single-spacing bipolar montage.** Following the paper, adjacent contacts on the same shaft are subtracted (`Ch1-Ch2`, `Ch2-Ch3`, …). This cancels the common reference and localises the signal. Because the recordings were loaded in **clean** mode, the channel set is already trimmed to the curated contacts, so the montage simply pairs adjacent clean contacts. Intuitively, subtracting a contact from its immediate neighbour cancels whatever the two share (the common reference and distant sources) and keeps the activity that is *local* to that pair.

**4.3 Segmentation.** We cut each recording into non-overlapping 3-second segments and draw 20 of them at random, with a fixed seed for reproducibility.

> The authors in (Stergiadis et al., 2026) have studied HFO-contaminated and HFO-free segments separately, after manual detection of HFO events. In this notebook we do not make such a differentiation, and we choose the twenty 3-second segments randomly, without accounting for HFO presence.

## 4. Running the preprocessing pipeline for all subjects

The loop below is **idempotent**: subjects whose segments already exist are skipped, so it is safe to
re-run after an interruption. The `local_utils` calls carry out steps 4.1–4.3 (notch, bipolar montage, and random
segmentation) and save the result; the data fetching is done in the open with CobraBox. Just as we loaded `sub-01` on its own above, the loop now loads each subject in turn and applies the same steps.

NOTE: When the clean option is used when loading the zurich iEEG data, the "Missing monopolar channels" are also printed, which are the electrodes trimmed from the original dataset

In [ ]:
#### !!!! ?  should we do the notch filtering and bipolar montage separately from the segment extraction?

for subject_id in SUBJECTS:
    out_path = SEGMENTS_DIR / f"{subject_id}_segments.nc"
    if out_path.exists():
        print(f"{subject_id}: already preprocessed — skipping")
        continue

    # 1. Fetch the clean recordings (all runs) for this subject
    print(f"{subject_id}: loading…", end=" ", flush=True)
    ds = cb.load_dataset(DATASET_NAME, subset=[subject_id])

    # 2. Notch-filter, build the single-spacing bipolar montage, and cut
    #    20 random 3-second segments (see local_utils.py for each step).
    segments = local_utils.extract_segments(ds, subject_id, patient_info)

    # 3. Save as an (segment x space x time) NetCDF for notebook 3
    local_utils.save_segments(segments, subject_id)
    print(f"saved {len(segments)} segments → {out_path}")

    del ds, segments

## 5. Verifying the output

A quick spot-check on `sub-01`: the number of segments and channels, the samples per segment, and how
many resected and non-resected electrodes does the patient has.

In [ ]:
import xarray as xr

subject_id = "sub-01"
segments = xr.open_dataarray(SEGMENTS_DIR / f"{subject_id}_segments.nc")
resected = set(patient_info[subject_id]["resected"])
channels = list(segments.coords["space"].values)

print(f"Subject:              {subject_id}")
print(f"Segments:             {segments.sizes['segment']} (target {N_SEGMENTS})")
print(f"Bipolar channels:     {len(channels)}")
print(f"Samples per segment:  {segments.sizes['time']}")
print(f"Sampling rate:        {segments.attrs.get('sampling_rate', 'n/a')} Hz")
print(f"Resected pairs kept:  {sum(ch in resected for ch in channels)} / {len(resected)}")

## 6. Directed connectivity (DTF)

For each subject we load the saved segments and compute a **directed connectivity matrix** with
CobraBox. Directed connectivity asks, for every pair of channels, *who drives whom*: the influence
from channel *j* to channel *i* is not the same as from *i* to *j*. We use the **Directed Transfer
Function (DTF)**, estimated from a vector-autoregressive model of order `VAR_ORDER` (10), averaged
over the segments for each frequency band (deltha, theta, alpha, beta, low gamma, high gamma, ripples, fast ripples).

We compute it directly with CobraBox's DTF feature (`cb.feature.DirectedTransferFunction`): apply it
to each 3-second segment, average the matrices across the 20 segments, then average all the bins within each frequency band. The
result per subject is one `(space_to × space_from)` matrix per band, which we save so the analysis
below can be re-run without recomputing.

### Configuration for the connectivity computation

We point CobraBox at the same local `data/` folder used by notebook 2, list the subjects, and choose
the directed-connectivity method. The eight frequency bands and the VAR model order are taken from
`local_utils`, so notebooks 2 and 3 stay in sync.

In [ ]:
# # Point CobraBox at the local data folder (the segments from notebook 2 live here)
# cb.set_dataset_dir(Path(".") / "data", persist=False)

# SUBJECTS = [f"sub-{i:02d}" for i in range(1, 21)]

# Label used only in the names of the saved connectivity/strength files
CONNECTIVITY_METHOD = "dtf"

# Eight frequency bands, taken from local_utils (identical to the study)
BANDS = local_utils.BANDS

# Patient metadata: outcome (ILAE grade) and the list of resected bipolar pairs
patient_info = local_utils.load_patient_info()


def outcome_group(subject_id):
    """Good outcome = ILAE 1, poor outcome = ILAE 2-6 (adjust to your metadata format)."""
    grade = str(patient_info[subject_id]["outcome"]).upper().replace(" ", "")
    return "good" if grade in ("ILAE1", "1") else "poor"

In [ ]:
# Directed Transfer Function feature (VAR model order taken from local_utils)
dtf = cb.feature.DirectedTransferFunction(var_order=local_utils.VAR_ORDER, n_freqs=1000)

for subject_id in SUBJECTS:
    # Load this subject's segments (saved by notebook 2)
    segments = local_utils.load_segments(subject_id)

    # DTF per segment -> average across segments -> average within each frequency band
    per_segment = [dtf.apply(item).data for item in segments]
    conn = xr.concat(per_segment, dim="segment").mean("segment")
    avg_conn = {
        band: conn.sel(frequency=slice(low, high)).mean("frequency")
        for band, (low, high) in BANDS.items()
    }

    # Save the band-averaged matrices for reuse
    local_utils.save_connectivity(avg_conn, subject_id, method=CONNECTIVITY_METHOD)
    print(f"{subject_id}: DTF connectivity saved")

To make this concrete, let us look at one example: the eight band-averaged adjacency matrices computed above for the example subject (`sub-01`).

In [ ]:
# --- Example adjacency (connectivity) matrices for sub-01, all eight bands ---
# The DTF result above is, per band, a channel-by-channel directed connectivity
# (adjacency) matrix. Here we display all eight bands for the example subject.

example_subject = "sub-01"
conn = local_utils.load_connectivity(example_subject, method=CONNECTIVITY_METHOD)

# Shared colour scale across bands for comparability
vmax = max(float(conn[band].max()) for band in BANDS)

fig, axes = plt.subplots(4, 2, figsize=(9, 16), constrained_layout=True)
for ax, band in zip(axes.flatten(), BANDS):
    im = ax.imshow(conn[band].values, cmap="viridis", vmin=0, vmax=vmax, aspect="equal")
    ax.set_title(band)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.6, label="directed connectivity")
fig.suptitle(
    f"{CONNECTIVITY_METHOD.upper()} adjacency matrices \u2014 {example_subject}  (rows: to, cols: from)",
    fontsize=13,
)
plt.show()

## 7. Inward and outward strength

From each connectivity matrix we compute two numbers **per contact and per band**:

- **inward strength** — the average influence the contact *receives* (the mean of its incoming edges);
- **outward strength** — the average influence the contact *sends* (the mean of its outgoing edges).

Self-connections (the matrix diagonal) are ignored. Finally, following the study, we rescale each
measure to the range `[0, 1]` **within each patient and band** (min-max normalisation), so that values
are comparable across patients with different numbers of electrodes. This is short and central to the analysis, so we write it out here rather than placing it in
`local_utils`.

In [ ]:
def inward_outward(matrix):
    """Inward and outward strength from one (space_to x space_from) matrix.

    Entry [i, j] is the influence from channel j to channel i.
    inward[i]  = mean over j of [i, j]   (influence received)
    outward[j] = mean over i of [i, j]   (influence sent)
    """
    m = matrix.transpose("space_to", "space_from").values.astype(float).copy()
    np.fill_diagonal(m, np.nan)  # ignore self-connections
    inward = np.nanmean(m, axis=1)
    outward = np.nanmean(m, axis=0)
    return inward, outward


def minmax(x):
    """Rescale a vector to [0, 1]; a flat vector becomes all zeros."""
    x = np.asarray(x, dtype=float)
    lo, hi = np.nanmin(x), np.nanmax(x)
    return (x - lo) / (hi - lo) if hi > lo else np.zeros_like(x)


# Collect one row per (subject, band, contact) into a tidy table
rows = []
for subject_id in SUBJECTS:
    conn = local_utils.load_connectivity(subject_id, method=CONNECTIVITY_METHOD)
    resected = set(patient_info[subject_id]["resected"])
    grp = outcome_group(subject_id)

    for band, matrix in conn.items():
        inward, outward = inward_outward(matrix)
        inward, outward = minmax(inward), minmax(outward)
        channels = matrix.coords["space_to"].values
        for ch, in_s, out_s in zip(channels, inward, outward):
            rows.append(
                {
                    "subject": subject_id,
                    "outcome": grp,
                    "band": band,
                    "channel": ch,
                    "region": "inside" if ch in resected else "outside",
                    "in_strength": float(in_s),
                    "out_strength": float(out_s),
                }
            )

strength_df = pd.DataFrame(rows)
print(f"{len(strength_df)} rows  ({strength_df['subject'].nunique()} subjects)")
print(strength_df.head())

# save the strength table for reuse in the following computation notebook
(local_utils.ANALYSIS_DIR).mkdir(parents=True, exist_ok=True)
strength_df.to_csv(local_utils.ANALYSIS_DIR / f"strength_{CONNECTIVITY_METHOD}.csv", index=False)

## 6. Where to go next

**Next:** `03_analysis.ipynb` loads these segments, computes **directed connectivity (DTF)** with
CobraBox, derives **inward and outward nodal strength**, and tests whether any of the two differ
inside vs. outside the resection separately in good and poor outcome patients, reproducing the main result of (Stergiadis et. al., 2026).